# BERT 微调用于文本分类（HuggingFace）

- **作者**：邓涵丹
- **日期**：2026-07-30
- **适用周次**：Week 4
- **分类**：NLP / Transformers / PyTorch
- **关键词**：BERT、微调、HuggingFace、文本分类

## 学习目标

1. 使用 HuggingFace 加载预训练的 BERT 模型和分词器。
2. 掌握 AdamW 优化器和带 warmup 的学习率调度。
3. 在 IMDB 数据集上微调 BERT，并评估性能。

## 1. 背景与问题

BERT（Bidirectional Encoder Representations from Transformers）通过在大规模语料上预训练，习得了丰富的语言知识。通过微调，可以轻松适应下游任务，取得优异效果。本 Notebook 将使用 `bert-base-uncased` 对 IMDB 电影评论进行情感分类。

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
import torch
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup

from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import sys
import re
from collections import Counter
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score

print("PyTorch version:", torch.__version__)
print("Transformers version:", __import__('transformers').__version__)

PyTorch version: 1.13.0+cu117
Transformers version: 4.46.3


## 2. 数据准备

使用 `datasets` 库加载 IMDB，并进行 tokenization。

In [2]:
dataset = load_dataset('imdb')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(['text'])
tokenized_datasets = tokenized_datasets.rename_column('label', 'labels')
tokenized_datasets.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

train_dataset = tokenized_datasets['train']
test_dataset = tokenized_datasets['test']

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

## 3. 模型加载与优化器配置

使用 `BertForSequenceClassification`，并设置 AdamW 优化器（带权重衰减）以及 warmup 调度器。

In [3]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# AdamW with weight decay
no_decay = ['bias', 'LayerNorm.weight']
optimizer_params = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
     'weight_decay': 0.0}
]
optimizer = AdamW(optimizer_params, lr=2e-5)

num_epochs = 3
num_training_steps = num_epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_training_steps // 10,
    num_training_steps=num_training_steps
)

criterion = torch.nn.CrossEntropyLoss()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/root/miniconda3/lib/python3.8/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


## 3.1 AdamW vs. Adam：L2 正则化与权重衰减的“脱钩”

在深度学习中，为了防止过拟合，通常使用 L2 正则化（在损失函数后加 `λ||θ||²`）。但在 Adam 优化器中，L2 正则化实现得非常“不干净”。

**Adam 中 L2 正则化的问题**：
在 Adam 中，L2 正则项的梯度会除以梯度的历史平方根（`v_t` 的修正）。这意味着，**梯度变化剧烈的参数（v_t 大）会受到较小的正则化约束**，而**梯度平稳的参数会受到极大的正则化惩罚**。这导致了正则化力度在不同参数间严重失衡，使得 Adam 的泛化能力常常不如 SGD。

**AdamW 的解决方案**：
AdamW 直接将权重衰减（Weight Decay）从损失函数中剥离出来，不再除以 `v_t`。它在更新参数时直接减去 `η * λ * θ`（公式：`θ = θ - η * ∇L - η * λ * θ`）。

**为什么设置 `weight_decay=0.01` 给大部分参数，但对 `bias` 和 `LayerNorm` 设为 0？**
- **Bias（偏置）**：只影响平移，不影响拟合曲线的弯曲程度，对过拟合影响极小。
- **LayerNorm**：其可训练参数（`scale` 和 `shift`）也是用于调整分布的平稳性，对幅度不敏感。
对这两类参数做权重衰减会产生不必要的偏差，因此通常将其排除在外。

## 3.2 为什么需要 Warmup？

BERT 的微调通常采用较小的学习率（如 2e-5），并且使用 Warmup（预热）策略。

**原因如下**：
1. **预训练与微调的差异**：预训练模型（bert-base）在预训练阶段见过海量数据，但微调数据（如 IMDB）是特定领域的。刚开始训练时，模型的注意力分布极不稳定。
2. **梯度爆炸风险**：如果一开始就使用较大的学习率（如 2e-5），巨大的梯度会瞬间破坏预训练学到的宝贵权重。
3. **Warmup 的作用**：
   - 在训练的**前 10%** 的步骤中，学习率从 0 线性增加到 2e-5。
   - 这段时间模型在做“适应性调整”，仅以极小的步长微调权重的方向。
   - 当模型逐渐适应新数据的分布后，再逐步退火（线性衰减）学习率，以找到最优点。

通俗讲：**刚从冰面（预训练）下来的车（模型），必须先低速缓慢启动（Warmup），等轮胎（注意力权重）适应了沥青路面（下游任务），再加速冲刺（正常学习率）。**

## 3.3 解释 `outputs.logits`

在使用 `BertForSequenceClassification` 时，传入 `input_ids` 和 `attention_mask`，模型输出的是一个 `SequenceClassifierOutput` 对象。

`outputs.logits` 的结构为 `(Batch_Size, Num_Labels)`。
- BERT 在内部会自动提取 `[CLS]` 位置（即序列的第一个 Token）的输出向量。
- 然后将这个 768 维的向量通过一个随机初始化的分类头（`nn.Linear(768, 2)`）映射成 2 维的输出（即 logits）。
- 这个 `[CLS]` 标记（Classification Token）经过 BERT 多层双向交互后，其向量被认为聚合了整个句子的语义信息，因此专门用于分类任务。

注意：在代码中没有手动提取 `[CLS]`，因为 HuggingFace 的 `BertForSequenceClassification` 已经做好了这一步。在训练时仅需对 `logits` 计算 `CrossEntropyLoss` 即可。

## 4. 训练与评估

In [4]:
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    for batch in tqdm(loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), accuracy_score(all_labels, all_preds)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return accuracy_score(all_labels, all_preds)

train_losses, train_accs, test_accs = [], [], []
for epoch in range(1, num_epochs+1):
    loss, acc = train_epoch(model, train_loader, optimizer, scheduler)
    test_acc = evaluate(model, test_loader)
    train_losses.append(loss)
    train_accs.append(acc)
    test_accs.append(test_acc)
    print(f"Epoch {epoch}: Train Loss={loss:.4f}, Train Acc={acc:.4f}, Test Acc={test_acc:.4f}")

Evaluating: 100%|██████████| 1563/1563 [00:22<00:00, 70.11it/s]


Epoch 1: Train Loss=0.3763, Train Acc=0.8239, Test Acc=0.8742


Evaluating: 100%|██████████| 1563/1563 [00:22<00:00, 69.98it/s]


Epoch 2: Train Loss=0.2036, Train Acc=0.9190, Test Acc=0.8909


Evaluating: 100%|██████████| 1563/1563 [00:22<00:00, 69.86it/s]

Epoch 3: Train Loss=0.0941, Train Acc=0.9672, Test Acc=0.8906


## 5. 总结与参考

- 使用 HuggingFace 微调 BERT 极为便捷，只需少量代码即可获得高准确率。
- AdamW 和 warmup 是 BERT 微调的标准配置。
- BERT 强大的迁移能力使其成为许多 NLP 任务的首选。

**参考资料**：
- [BERT: Pre-training of Deep Bidirectional Transformers](https://arxiv.org/abs/1810.04805)
- [HuggingFace 微调教程](https://huggingface.co/docs/transformers/training)